# CasCrop: Full Experiment Pipeline
**Crop Waste as Economic Contagion via Graph Neural Networks**

Set `QUICK_TEST` below, then **Runtime > Run All**.

In [ ]:
#@title Configuration
QUICK_TEST = True  #@param {type:"boolean"}
SEEDS = [42, 123, 456] if QUICK_TEST else [42, 123, 456, 789, 1024]
EPOCHS = 20 if QUICK_TEST else 200
PATIENCE = 10 if QUICK_TEST else 20
BATCH_SIZE = 1024 if QUICK_TEST else 512
print(f"{'QUICK' if QUICK_TEST else 'FULL'}: {len(SEEDS)} seeds, {EPOCHS} epochs")

## 1. Setup

In [ ]:
import torch
print(f'PyTorch {torch.__version__}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_mem/1e9:.1f}GB)')
    if torch.cuda.get_device_properties(0).total_mem < 8e9:
        BATCH_SIZE = min(BATCH_SIZE, 256)
else:
    print('NO GPU - Runtime > Change runtime type > T4')

In [ ]:
import os, subprocess
if not os.path.exists('CasCrop'):
    !git clone https://github.com/keshavkrishnan08/CasCrop.git
%cd CasCrop
!pip install -q pandas pyarrow scipy scikit-learn statsmodels seaborn tqdm pyyaml openpyxl 2>&1 | tail -1
import sys, json, time, shutil, re
import numpy as np, pandas as pd
from pathlib import Path
sys.path.insert(0, 'src')
for d in ['data/raw/rma','data/raw/nass','data/raw/weather','data/raw/prices','data/raw/geographic','data/processed','data/graphs','checkpoints','results','paper/figures','paper/tables']:
    os.makedirs(d, exist_ok=True)
print('OK')

In [ ]:
#@title Mount Google Drive (optional)
SAVE_TO_DRIVE = False; DRIVE_PATH = ''
try:
    from google.colab import drive; drive.mount('/content/drive')
    DRIVE_PATH = '/content/drive/MyDrive/CasCrop_Results'
    os.makedirs(f'{DRIVE_PATH}/checkpoints', exist_ok=True)
    for f in Path(f'{DRIVE_PATH}/checkpoints').glob('*.pt'):
        dst = Path(f'checkpoints/{f.name}')
        if not dst.exists(): shutil.copy2(f, dst)
    SAVE_TO_DRIVE = True; print(f'Drive: {DRIVE_PATH}')
except: print('No Drive')
def backup():
    if not SAVE_TO_DRIVE: return
    for d in ['results','checkpoints','paper/figures','paper/tables']:
        if not os.path.exists(d): continue
        dst = f'{DRIVE_PATH}/{d}'; os.makedirs(dst, exist_ok=True)
        for f in Path(d).glob('*'):
            if f.is_file(): shutil.copy2(f, f'{dst}/{f.name}')

## 2. Load Data
Auto-downloads pre-processed data (54MB) from GitHub Releases. No manual steps needed.

In [ ]:
#@title Load Pre-processed Data (automatic)
import os
from pathlib import Path

if Path('data/processed/features.parquet').exists() and Path('data/graphs/combined_graph.npz').exists():
    print('Data already present.')
else:
    DATA_URL = 'https://github.com/keshavkrishnan08/CasCrop/releases/download/v0.1-data/cascrop_processed.tar.gz'
    print('Downloading pre-processed data (54MB)...')
    !wget -q --show-progress -O cascrop_processed.tar.gz {DATA_URL}
    !tar xzf cascrop_processed.tar.gz
    !rm cascrop_processed.tar.gz
    print('Extracted.')

import pandas as pd
f = pd.read_parquet('data/processed/features.parquet')
l = pd.read_parquet('data/processed/labels.parquet')
print(f'{len(f):,} samples, {f["fips"].nunique()} counties, {l["waste"].mean():.1%} waste rate')


## 2. Download Data (~10 min)

In [ ]:
import requests, zipfile, gzip
def dl(url, path, desc='', min_size=100, retries=3):
    path = Path(path)
    if path.exists() and path.stat().st_size > min_size:
        with open(path,'rb') as f: h = f.read(20)
        if not (h.strip().startswith(b'<!') or h.strip().startswith(b'<html')): return True
        path.unlink()
    path.parent.mkdir(parents=True, exist_ok=True)
    hdrs = {'User-Agent':'Mozilla/5.0 (Windows NT 10.0; Win64; x64) CasCrop/1.0'}
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=hdrs, timeout=300)
            if r.status_code in (403,404): print(f'  {r.status_code}: {desc}'); return False
            r.raise_for_status()
            if r.content[:20].strip().startswith(b'<!'): return False
            path.write_bytes(r.content)
            if path.stat().st_size < min_size: path.unlink(); return False
            print(f'  OK {path.name} ({path.stat().st_size:,}B)'); return True
        except Exception as e:
            if attempt < retries-1: time.sleep(2**attempt)
    print(f'  FAIL: {desc}'); return False

In [ ]:
%%time
print('=== RMA ===')
rma_ok = 0
for yr in range(2015, 2026):
    txt = Path(f'data/raw/rma/colsom_{yr}.txt')
    if txt.exists() and txt.stat().st_size > 1000: rma_ok += 1; continue
    zp = Path(f'data/raw/rma/colsom_{yr}.zip')
    if dl(f'https://pubfs-rma.fpac.usda.gov/pub/Web_Data_Files/Summary_of_Business/cause_of_loss/colsom_{yr}.zip', zp, f'RMA {yr}', 1000):
        try:
            with zipfile.ZipFile(zp) as zf: zf.extractall('data/raw/rma/')
            rma_ok += 1
        except: print(f'  Bad zip {yr}')
print(f'RMA: {rma_ok} years')
if rma_ok < 5: raise RuntimeError(f'Need >=5 RMA years, got {rma_ok}')

In [ ]:
%%time
print('=== Geographic ===')
dl('https://www2.census.gov/geo/docs/reference/county_adjacency.txt','data/raw/geographic/county_adjacency.txt','Adjacency')
for yr in ['2023','2022','2021']:
    zp = Path(f'data/raw/geographic/gaz_{yr}.zip')
    if dl(f'https://www2.census.gov/geo/docs/maps-data/data/gazetteer/{yr}_Gazetteer/{yr}_Gaz_counties_national.zip', zp, f'Gaz {yr}'):
        with zipfile.ZipFile(zp) as zf: zf.extractall('data/raw/geographic/')
        break

In [ ]:
%%time
print('=== Prices ===')
ok = True
for crop, sid in [('corn','PMAIZMTUSDM'),('wheat','PWHEAMTUSDM'),('soybeans','PSOYBUSDM')]:
    if not dl(f'https://fred.stlouisfed.org/graph/fredgraph.csv?id={sid}', f'data/raw/prices/{crop}_prices.csv', f'FRED {crop}'): ok = False
if not ok:
    print('FRED blocked. World Bank fallback...')
    if dl('https://thedocs.worldbank.org/en/doc/5d903e848db1d1b83e0ec8f744e55570-0350012021/related/CMO-Historical-Data-Monthly.xlsx','data/raw/prices/wb.xlsx','WB',10000):
        try:
            wb = pd.read_excel('data/raw/prices/wb.xlsx', sheet_name='Monthly Prices', skiprows=4)
            for crop, hint in [('corn','MAIZE'),('wheat','WHEAT'),('soybeans','SOYBEAN')]:
                cols = [c for c in wb.columns if hint.lower() in c.lower()]
                if cols:
                    d = wb[['Unnamed: 0', cols[0]]].dropna(); d.columns = ['DATE', crop.upper()]
                    d.to_csv(f'data/raw/prices/{crop}_prices.csv', index=False)
                    print(f'  {crop}: {len(d)} from WB')
        except Exception as e: print(f'  WB error: {e}')
    # BLS API last resort
    for crop, sid in [('corn','WPU012202'),('wheat','WPU0121'),('soybeans','WPU01830101')]:
        p = Path(f'data/raw/prices/{crop}_prices.csv')
        if p.exists() and p.stat().st_size > 100: continue
        try:
            r = requests.get(f'https://data.bls.gov/timeseries/{sid}?output_type=json', timeout=30)
            if r.ok:
                rows = [{'DATE':f"{dp['year']}-{dp['period'][1:]}-01",crop.upper():dp['value']} for s in r.json().get('Results',{}).get('series',[]) for dp in s.get('data',[])]
                if rows: pd.DataFrame(rows).to_csv(p, index=False); print(f'  {crop}: {len(rows)} from BLS')
        except: pass
for crop in ['corn','wheat','soybeans']:
    p = Path(f'data/raw/prices/{crop}_prices.csv')
    print(f'  {crop}: {len(pd.read_csv(p)):,}' if p.exists() and p.stat().st_size>50 else f'  {crop}: MISSING')

In [ ]:
%%time
print('=== NOAA Climate ===')
FB='20260305'
try:
    r=requests.get('https://www.ncei.noaa.gov/pub/data/cirs/climdiv/',headers={'User-Agent':'Mozilla/5.0'},timeout=30)
    dates=re.findall(r'climdiv-tmaxcy-v[\d.]+-([\d]+)',r.text); ds=max(dates) if dates else FB
except: ds=FB
print(f'Date: {ds}')
base='https://www.ncei.noaa.gov/pub/data/cirs/climdiv/'
for var,code in [('tmax','tmaxcy'),('tmin','tmincy'),('tavg','tmpccy'),('precip','pcpncy'),('pdsi','pdsicy'),('cdd','cddccy'),('hdd','hddccy')]:
    fn=f'climdiv_{var}_county.txt'
    ok=dl(f'{base}climdiv-{code}-v1.0.0-{ds}',f'data/raw/weather/{fn}',fn,10000)
    if not ok and ds!=FB: dl(f'{base}climdiv-{code}-v1.0.0-{FB}',f'data/raw/weather/{fn}',f'{fn}(fb)',10000)

In [ ]:
%%time
print('=== NASS ===')
nass_clean = Path('data/raw/nass/all_crops_county_annual.csv')
if nass_clean.exists() and nass_clean.stat().st_size > 10000:
    print(f'Done: {len(pd.read_csv(nass_clean)):,}')
else:
    nass_gz = Path('data/raw/nass/qs.crops.txt.gz')
    if not (nass_gz.exists() and nass_gz.stat().st_size > 1e8):
        from datetime import datetime, timedelta
        for days in range(0, 90, 7):
            dt = datetime.now() - timedelta(days=days)
            if dl(f'https://www.nass.usda.gov/datasets/qs.crops_{dt.strftime("%Y%m%d")}.txt.gz', nass_gz, f'NASS {dt.strftime("%Y%m%d")}', int(1e6)): break
        if not (nass_gz.exists() and nass_gz.stat().st_size > 1e6):
            print('NASS FAILED. Go to https://www.nass.usda.gov/datasets/ and upload qs.crops_*.txt.gz')
    if nass_gz.exists() and nass_gz.stat().st_size > 1e6:
        print(f'Extracting {nass_gz.stat().st_size/1e9:.1f}GB...')
        tc={'CORN','SOYBEANS','WHEAT'};ts={'YIELD','PRODUCTION','AREA PLANTED','AREA HARVESTED'}
        filt=Path('data/raw/nass/filtered.tsv');n=k=0
        with gzip.open(nass_gz,'rt',encoding='latin-1') as fin,open(filt,'w') as fout:
            hdr=fin.readline().strip();fout.write(hdr+'\n');cols=hdr.split('\t')
            ci,si,ai=cols.index('COMMODITY_DESC'),cols.index('STATISTICCAT_DESC'),cols.index('AGG_LEVEL_DESC')
            for line in fin:
                n+=1;p=line.strip().split('\t')
                if len(p)<=max(ci,si,ai):continue
                if p[ci].strip() in tc and p[si].strip() in ts and p[ai].strip()=='COUNTY':fout.write(line);k+=1
                if n%5_000_000==0:print(f'  {n:,}/{k:,}')
        print(f'  {k:,} records')
        df=pd.read_csv(filt,sep='\t',low_memory=False,dtype={'STATE_ANSI':str,'COUNTY_ANSI':str})
        df['FIPS']=df['STATE_ANSI'].str.zfill(2)+df['COUNTY_ANSI'].str.zfill(3)
        df=df[df['YEAR']>=2008]
        df['V']=pd.to_numeric(df['VALUE'].astype(str).str.replace(',','').str.strip(),errors='coerce')
        pv=df.pivot_table(index=['FIPS','STATE_ANSI','STATE_ALPHA','COUNTY_ANSI','YEAR','COMMODITY_DESC'],columns='STATISTICCAT_DESC',values='V',aggfunc='first').reset_index()
        pv.columns.name=None
        pv.rename(columns={'COMMODITY_DESC':'crop','AREA HARVESTED':'area_harvested_acres','AREA PLANTED':'area_planted_acres','PRODUCTION':'production_bu','YIELD':'yield_bu_per_acre'},inplace=True)
        pv['waste_proxy']=((pv['area_planted_acres']-pv['area_harvested_acres'])/pv['area_planted_acres']).clip(0)
        pv.to_csv(nass_clean,index=False);print(f'  Saved {len(pv):,}')

## 3. Process + Graphs

In [ ]:
%%time
if not Path('data/processed/features.parquet').exists():
    !python scripts/02_process_data.py --threshold 100000
else: print('Processed')
if not Path('data/graphs/combined_graph.npz').exists():
    !python scripts/03_build_graphs.py --top-k 20
else: print('Graphs OK')
features=pd.read_parquet('data/processed/features.parquet')
labels=pd.read_parquet('data/processed/labels.parquet')
with open('data/processed/splits.json') as f:splits=json.load(f)
with open('data/processed/feature_groups.json') as f:groups=json.load(f)
print(f'{len(features):,} samples, {labels["waste"].mean():.1%} waste')
backup()

## 4. Main Ablation

In [ ]:
%%time
models=['local_only','local_econ','geo_gat','symmetric_ecmp','cascrop']
ss=' '.join(str(s) for s in SEEDS);t0=time.time()
for i,m in enumerate(models):
    print(f'\n{"="*50}\n[{i+1}/{len(models)}] {m}\n{"="*50}')
    try:
        r=subprocess.run(f'python scripts/04_train_all.py --models {m} --seeds {ss} --epochs {EPOCHS} --patience {PATIENCE} --batch-size {BATCH_SIZE} --gpu 0 --resume',shell=True,capture_output=True,text=True,timeout=7200)
        for line in r.stdout.strip().split('\n')[-10:]:print(line)
        if r.returncode!=0 and r.stderr:print(f'ERR:{r.stderr[-200:]}')
    except subprocess.TimeoutExpired:print('Timeout')
    except Exception as e:print(f'Error:{e}')
    backup()
    print(f'ETA:{(time.time()-t0)/(i+1)*(len(models)-i-1)/60:.0f}min')
    if torch.cuda.is_available():torch.cuda.empty_cache()

In [ ]:
with open('results/training_results.json') as f:res=json.load(f)
df=pd.DataFrame(res)
for m in models:
    d=df[df['model']==m]
    if len(d):print(f'{m:<20} AUC={d["test_auc_roc"].mean():.3f}+/-{d["test_auc_roc"].std():.3f}')

## 5. Extra Experiments

In [ ]:
%%time
# Graph perturbation
print('=== Perturbation ===')
g=np.load('data/graphs/combined_graph.npz');np.random.seed(42)
np.savez('data/graphs/shuffled.npz',edge_index=np.array([g['edge_index'][0],np.random.permutation(g['edge_index'][1])]),edge_weight=g['edge_weight'])
ss3=' '.join(str(s) for s in SEEDS[:3])
subprocess.run(f'python scripts/04_train_all.py --models cascrop --seeds {ss3} --epochs {EPOCHS} --patience {PATIENCE} --batch-size {BATCH_SIZE} --gpu 0 --graph data/graphs/shuffled.npz',shell=True,capture_output=True,text=True,timeout=3600)
shutil.copy('results/training_results.json','results/perturbation.json')
pr=pd.DataFrame(json.load(open('results/perturbation.json')))
print(f'Shuffled:{pr["test_auc_roc"].mean():.3f}+/-{pr["test_auc_roc"].std():.3f}')
if torch.cuda.is_available():torch.cuda.empty_cache()

In [ ]:
%%time
# Edge ablation
print('=== Edge Ablation ===')
from scipy import sparse
geo=sparse.load_npz('data/graphs/adjacency_geo.npz')
def sp2npz(mat,path,k=20):
    d=mat.toarray();n=d.shape[0];r,c,v=[],[],[]
    for i in range(n):
        nz=np.where(d[i]>0)[0]
        if len(nz)==0:continue
        top=nz[np.argsort(d[i,nz])[-k:]]
        for j in top:r.append(i);c.append(j);v.append(d[i,j])
    np.savez(path,edge_index=np.array([r,c]),edge_weight=np.array(v))
sp2npz(geo,'data/graphs/geo_only.npz')
comm=(sparse.load_npz('data/graphs/adjacency_commodity_corn.npz')+sparse.load_npz('data/graphs/adjacency_commodity_soybeans.npz')+sparse.load_npz('data/graphs/adjacency_commodity_wheat.npz'))/3
sp2npz(comm,'data/graphs/comm_only.npz')
for nm,gp in [('geo_only','data/graphs/geo_only.npz'),('comm_only','data/graphs/comm_only.npz')]:
    subprocess.run(f'python scripts/04_train_all.py --models cascrop --seeds {ss3} --epochs {EPOCHS} --patience {PATIENCE} --batch-size {BATCH_SIZE} --gpu 0 --graph {gp}',shell=True,capture_output=True,text=True,timeout=3600)
    rd=pd.DataFrame(json.load(open('results/training_results.json')))
    print(f'  {nm}:{rd["test_auc_roc"].mean():.3f}+/-{rd["test_auc_roc"].std():.3f}')
    if torch.cuda.is_available():torch.cuda.empty_cache()
backup()

## 6. Evaluation

In [ ]:
subprocess.run(f'python scripts/04_train_all.py --seeds {ss} --epochs {EPOCHS} --patience {PATIENCE} --batch-size {BATCH_SIZE} --gpu 0 --resume',shell=True,capture_output=True,text=True,timeout=600)
!python scripts/05_evaluate_and_publish.py

In [ ]:
from evaluation.statistical_tests import paired_ttest_across_seeds
with open('results/training_results.json') as f:res=json.load(f)
df=pd.DataFrame(res);ca=sorted(df[df['model']=='cascrop']['test_auc_roc'].tolist())
for m in ['local_only','local_econ','geo_gat','symmetric_ecmp']:
    ma=sorted(df[df['model']==m]['test_auc_roc'].tolist())
    if len(ma)!=len(ca):continue
    t=paired_ttest_across_seeds(ca,ma)
    sig='***' if t['p_value']<.001 else '**' if t['p_value']<.01 else '*' if t['p_value']<.05 else 'ns'
    print(f'vs {m:<20} d={t["mean_diff"]:+.4f} p={t["p_value"]:.4f} {sig}')
c=df[df['model']=='cascrop']['test_auc_roc'].mean()
l=df[df['model']=='local_only']['test_auc_roc'].mean()
g=df[df['model']=='geo_gat']['test_auc_roc'].mean()
s=df[df['model']=='symmetric_ecmp']['test_auc_roc'].mean()
print(f'H1 +{c-l:.3f} H2 +{c-g:.3f} H3 +{c-s:.3f}')

In [ ]:
import matplotlib.pyplot as plt
mo=['local_only','local_econ','geo_gat','symmetric_ecmp','cascrop']
dn=['Local Only','Local+Econ','Geo GAT','Sym ECMP','CasCrop']
co=['#7f8c8d','#3498db','#e67e22','#9b59b6','#e74c3c']
ms=[df[df['model']==m]['test_auc_roc'].mean() for m in mo]
ss_=[df[df['model']==m]['test_auc_roc'].std() for m in mo]
fig,ax=plt.subplots(figsize=(7,3.5))
bars=ax.bar(range(5),ms,0.6,yerr=ss_,capsize=4,color=co,edgecolor='k',linewidth=.5)
for b,v in zip(bars,ms):ax.text(b.get_x()+.3,b.get_height()+.008,f'{v:.3f}',ha='center',fontsize=7)
ax.set_xticks(range(5));ax.set_xticklabels(dn,fontsize=8)
ax.set_ylim(.7,1);ax.set_ylabel('AUC-ROC');ax.grid(axis='y',alpha=.3)
ax.set_title('Ablation Results',fontweight='bold')
plt.tight_layout();fig.savefig('paper/figures/fig3_ablation.pdf',dpi=300);plt.show()

## 7. Download

In [ ]:
backup()
!tar czf /content/cascrop_results.tar.gz results/ paper/figures/ paper/tables/ checkpoints/
try:
    from google.colab import files;files.download('/content/cascrop_results.tar.gz')
except:print('Download from Files or Drive')
print('ALL DONE')